In [1]:
import torch
import torch.nn as nn

In [2]:
def conv_block(inp_filt, out_filt):
    conv = nn.Sequential(nn.Conv2d(inp_filt, out_filt, 3, padding=1),
                         nn.BatchNorm2d(out_filt),
                         nn.ReLU(inplace=True)
                         )
    return conv

In [3]:
class Encoder_Block(nn.Module):
    def __init__(self, inp_filt, out_filt):
        super().__init__()
        self.conv2k = nn.Sequential(
            conv_block(inp_filt, out_filt),
            conv_block(out_filt, out_filt)
        )
        self.x_2 = nn.MaxPool2d(2,2)

    def forward(self, x):
        skip = self.conv2k(x)
        out = self.x_2(skip)
        return skip, out

In [4]:
class Bro_UNET(nn.Module):
    def __init__(self, ifilt=3, nf=64):
        super().__init__()
        up_f = [128, 256, 512, 1024]
        d_f = [512, 256, 128, 64]
        self.enc_blk_1 = Encoder_Block(ifilt, nf) #x, x/2 .. x is skip, x/2 in input to next encoder block
        self.enc_blk_2 = Encoder_Block(nf, up_f[0]) #x/2, x/4
        self.enc_blk_3 = Encoder_Block(up_f[0], up_f[1]) #x/4, x/8
        self.enc_blk_4 = Encoder_Block(up_f[1], up_f[2]) #x/8, x/16

        self.bneck = conv_block(up_f[2], up_f[3]) #x/16
        self.bneck_x_2x = nn.ConvTranspose2d(up_f[3], d_f[0], (2,2), 2)

        self.dec_blk_enc_blk_3 = self.decoder_blk(d_f[0]) #512
        self.dec_blk_enc_blk_2 = self.decoder_blk(d_f[1]) #256
        self.dec_blk_enc_blk_1 = self.decoder_blk(d_f[2]) #128

        self.final_conv = nn.Sequential(
            conv_block(d_f[2], d_f[3]),
            nn.Conv2d(d_f[3], 1, 1),
            nn.Sigmoid()
            )

    def decoder_blk(self, inp_filt):
        conv_x_2x = nn.Sequential(
            conv_block(inp_filt*2, inp_filt),
            conv_block(inp_filt, inp_filt),
            nn.ConvTranspose2d(inp_filt, int(inp_filt/2), (2,2), 2)
            )
        return conv_x_2x

    def forward(self, x):
        s1, out = self.enc_blk_1(x)  #224, 112
        s2, out = self.enc_blk_2(out) #112, 56
        s3, out = self.enc_blk_3(out) #56, 28
        s4, out = self.enc_blk_4(out) #28, 14

        out = self.bneck(out)
        out = self.bneck_x_2x(out) #28

        out = torch.cat((s4, out), 1)
        out = self.dec_blk_enc_blk_3(out)
        out = torch.cat((s3, out), 1)
        out = self.dec_blk_enc_blk_2(out)
        out = torch.cat((s2, out), 1)
        out = self.dec_blk_enc_blk_1(out)
        out = torch.cat((s1, out), 1)
        out = self.final_conv(out)
        return out

In [5]:
inp = torch.ones(2,3,224, 224)
inp.shape

torch.Size([2, 3, 224, 224])

In [6]:
model = Bro_UNET()

In [7]:
!pip install torchinfo
from torchinfo import summary

In [8]:
out = model(inp)

In [9]:
out.shape

torch.Size([2, 1, 224, 224])

In [10]:
summary(model,input_size=(1,3,224,224))

Layer (type:depth-idx)                   Output Shape              Param #
Bro_UNET                                 [1, 1, 224, 224]          --
├─Encoder_Block: 1-1                     [1, 64, 224, 224]         --
│    └─Sequential: 2-1                   [1, 64, 224, 224]         --
│    │    └─Sequential: 3-1              [1, 64, 224, 224]         1,920
│    │    └─Sequential: 3-2              [1, 64, 224, 224]         37,056
│    └─MaxPool2d: 2-2                    [1, 64, 112, 112]         --
├─Encoder_Block: 1-2                     [1, 128, 112, 112]        --
│    └─Sequential: 2-3                   [1, 128, 112, 112]        --
│    │    └─Sequential: 3-3              [1, 128, 112, 112]        74,112
│    │    └─Sequential: 3-4              [1, 128, 112, 112]        147,840
│    └─MaxPool2d: 2-4                    [1, 128, 56, 56]          --
├─Encoder_Block: 1-3                     [1, 256, 56, 56]          --
│    └─Sequential: 2-5                   [1, 256, 56, 56]          --

In [11]:
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(nn.Conv2d(in_planes, in_planes // 16, 1, bias=False),
                               nn.ReLU(),
                               nn.Conv2d(in_planes // 16, in_planes, 1, bias=False))
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.avg_pool(x)
        print("1   ", avg_out.shape)
        avg_out = self.fc(avg_out)
        print("2   ", avg_out.shape)

        max_out = self.max_pool(x)
        print("3   ", max_out.shape)
        max_out = self.fc(max_out)
        print("4   ", max_out.shape)
        
        out = avg_out + max_out
        out = self.sigmoid(out)
        print("5   ", out.shape)
        out = out * x 
        return out

In [12]:
x = torch.ones(1,128, 16, 16)
print(x.shape)

torch.Size([1, 128, 16, 16])


In [13]:
ca = ChannelAttention(128)
summary(ca)

Layer (type:depth-idx)                   Param #
ChannelAttention                         --
├─AdaptiveAvgPool2d: 1-1                 --
├─AdaptiveMaxPool2d: 1-2                 --
├─Sequential: 1-3                        --
│    └─Conv2d: 2-1                       1,024
│    └─ReLU: 2-2                         --
│    └─Conv2d: 2-3                       1,024
├─Sigmoid: 1-4                           --
Total params: 2,048
Trainable params: 2,048
Non-trainable params: 0

In [14]:
out = ca(x)

1    torch.Size([1, 128, 1, 1])
2    torch.Size([1, 128, 1, 1])
3    torch.Size([1, 128, 1, 1])
4    torch.Size([1, 128, 1, 1])
5    torch.Size([1, 128, 1, 1])


In [15]:
out

tensor([[[[0.5448, 0.5448, 0.5448,  ..., 0.5448, 0.5448, 0.5448],
          [0.5448, 0.5448, 0.5448,  ..., 0.5448, 0.5448, 0.5448],
          [0.5448, 0.5448, 0.5448,  ..., 0.5448, 0.5448, 0.5448],
          ...,
          [0.5448, 0.5448, 0.5448,  ..., 0.5448, 0.5448, 0.5448],
          [0.5448, 0.5448, 0.5448,  ..., 0.5448, 0.5448, 0.5448],
          [0.5448, 0.5448, 0.5448,  ..., 0.5448, 0.5448, 0.5448]],

         [[0.4006, 0.4006, 0.4006,  ..., 0.4006, 0.4006, 0.4006],
          [0.4006, 0.4006, 0.4006,  ..., 0.4006, 0.4006, 0.4006],
          [0.4006, 0.4006, 0.4006,  ..., 0.4006, 0.4006, 0.4006],
          ...,
          [0.4006, 0.4006, 0.4006,  ..., 0.4006, 0.4006, 0.4006],
          [0.4006, 0.4006, 0.4006,  ..., 0.4006, 0.4006, 0.4006],
          [0.4006, 0.4006, 0.4006,  ..., 0.4006, 0.4006, 0.4006]],

         [[0.5213, 0.5213, 0.5213,  ..., 0.5213, 0.5213, 0.5213],
          [0.5213, 0.5213, 0.5213,  ..., 0.5213, 0.5213, 0.5213],
          [0.5213, 0.5213, 0.5213,  ..., 0

In [16]:
out.shape

torch.Size([1, 128, 16, 16])

In [17]:
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()

        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        print("1   ", avg_out.shape)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        print("2   ", max_out.shape)
        out = torch.cat([avg_out, max_out], dim=1)
        print("3   ", out.shape)
        out = self.conv1(out)
        print("4   ", out.shape)
        out = self.sigmoid(out)
        print("5   ", out.shape)
        out = out * x
        return out
        

In [18]:
sa = SpatialAttention()

In [19]:
out = sa(out)
out.shape

1    torch.Size([1, 1, 16, 16])
2    torch.Size([1, 1, 16, 16])
3    torch.Size([1, 2, 16, 16])
4    torch.Size([1, 1, 16, 16])
5    torch.Size([1, 1, 16, 16])


torch.Size([1, 128, 16, 16])

In [20]:
out

tensor([[[[0.2316, 0.2136, 0.2095,  ..., 0.2209, 0.2123, 0.2395],
          [0.2095, 0.1840, 0.1761,  ..., 0.1940, 0.1853, 0.2157],
          [0.2060, 0.1736, 0.1660,  ..., 0.1860, 0.1774, 0.2137],
          ...,
          [0.2074, 0.1924, 0.1779,  ..., 0.1895, 0.1930, 0.2279],
          [0.2078, 0.1980, 0.1735,  ..., 0.1906, 0.2020, 0.2256],
          [0.2295, 0.2306, 0.2122,  ..., 0.2242, 0.2373, 0.2504]],

         [[0.1702, 0.1571, 0.1540,  ..., 0.1624, 0.1561, 0.1760],
          [0.1540, 0.1353, 0.1295,  ..., 0.1426, 0.1363, 0.1586],
          [0.1515, 0.1276, 0.1221,  ..., 0.1367, 0.1304, 0.1571],
          ...,
          [0.1525, 0.1415, 0.1308,  ..., 0.1393, 0.1419, 0.1676],
          [0.1528, 0.1456, 0.1275,  ..., 0.1401, 0.1485, 0.1659],
          [0.1687, 0.1695, 0.1560,  ..., 0.1648, 0.1744, 0.1841]],

         [[0.2216, 0.2044, 0.2005,  ..., 0.2114, 0.2032, 0.2291],
          [0.2004, 0.1761, 0.1685,  ..., 0.1856, 0.1773, 0.2064],
          [0.1971, 0.1661, 0.1589,  ..., 0

In [21]:
def cbam(x):
    in_planes = x.shape[1]
    ca = ChannelAttention(in_planes)
    sa = SpatialAttention()
    x = ca(x)
    x = sa(x)
    return x

In [22]:
out = cbam(out)

1    torch.Size([1, 128, 1, 1])
2    torch.Size([1, 128, 1, 1])
3    torch.Size([1, 128, 1, 1])
4    torch.Size([1, 128, 1, 1])
5    torch.Size([1, 128, 1, 1])
1    torch.Size([1, 1, 16, 16])
2    torch.Size([1, 1, 16, 16])
3    torch.Size([1, 2, 16, 16])
4    torch.Size([1, 1, 16, 16])
5    torch.Size([1, 1, 16, 16])


In [27]:
out.shape

torch.Size([1, 128, 16, 16])

### Now you can insert cbam anywhere so long the in channels are multiple of 16

out = layer(out)

#### out = cbam(out)

..continue with rest

with input shape being equal shape

In [2]:
!pip install -U torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 167.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 19.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 34.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 144.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 21.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 60.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 162.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 54.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 45.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15

In [23]:
import torch
import torchvision
import torch.nn as nn

In [24]:
torch.hub.list("pytorch/vision", force_reload=True)

Downloading: "https://github.com/pytorch/vision/zipball/main" to /home/ec2-user/.cache/torch/hub/main.zip


['alexnet',
 'convnext_base',
 'convnext_large',
 'convnext_small',
 'convnext_tiny',
 'deeplabv3_mobilenet_v3_large',
 'deeplabv3_resnet101',
 'deeplabv3_resnet50',
 'densenet121',
 'densenet161',
 'densenet169',
 'densenet201',
 'efficientnet_b0',
 'efficientnet_b1',
 'efficientnet_b2',
 'efficientnet_b3',
 'efficientnet_b4',
 'efficientnet_b5',
 'efficientnet_b6',
 'efficientnet_b7',
 'efficientnet_v2_l',
 'efficientnet_v2_m',
 'efficientnet_v2_s',
 'fcn_resnet101',
 'fcn_resnet50',
 'get_model_weights',
 'get_weight',
 'googlenet',
 'inception_v3',
 'lraspp_mobilenet_v3_large',
 'maxvit_t',
 'mc3_18',
 'mnasnet0_5',
 'mnasnet0_75',
 'mnasnet1_0',
 'mnasnet1_3',
 'mobilenet_v2',
 'mobilenet_v3_large',
 'mobilenet_v3_small',
 'mvit_v1_b',
 'mvit_v2_s',
 'r2plus1d_18',
 'r3d_18',
 'raft_large',
 'raft_small',
 'regnet_x_16gf',
 'regnet_x_1_6gf',
 'regnet_x_32gf',
 'regnet_x_3_2gf',
 'regnet_x_400mf',
 'regnet_x_800mf',
 'regnet_x_8gf',
 'regnet_y_128gf',
 'regnet_y_16gf',
 'regnet_y_1

In [25]:
model = torch.hub.load("pytorch/vision", "resnet50", weights="ResNet50_Weights.IMAGENET1K_V1")

Using cache found in /home/ec2-user/.cache/torch/hub/pytorch_vision_main


In [26]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [27]:
summary(model, input_size=(1,3,224,224))

Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [1, 1000]                 --
├─Conv2d: 1-1                            [1, 64, 112, 112]         9,408
├─BatchNorm2d: 1-2                       [1, 64, 112, 112]         128
├─ReLU: 1-3                              [1, 64, 112, 112]         --
├─MaxPool2d: 1-4                         [1, 64, 56, 56]           --
├─Sequential: 1-5                        [1, 256, 56, 56]          --
│    └─Bottleneck: 2-1                   [1, 256, 56, 56]          --
│    │    └─Conv2d: 3-1                  [1, 64, 56, 56]           4,096
│    │    └─BatchNorm2d: 3-2             [1, 64, 56, 56]           128
│    │    └─ReLU: 3-3                    [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-4                  [1, 64, 56, 56]           36,864
│    │    └─BatchNorm2d: 3-5             [1, 64, 56, 56]           128
│    │    └─ReLU: 3-6                    [1, 64, 56, 56]           --
│ 

In [28]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [29]:
layer_0 = nn.Sequential(
    model.conv1,
    model.bn1,
    model.relu,
    model.maxpool
)

In [30]:
layer_1 = model.layer1
layer_2 = model.layer2
layer_3 = model.layer3
layer_4 = model.layer4

In [31]:
layers = [layer_0, layer_1, layer_2, layer_3, layer_4]
layers

[Sequential(
   (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
   (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   (2): ReLU(inplace=True)
   (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
 ),
 Sequential(
   (0): Bottleneck(
     (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
     (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
     (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (relu): ReLU(inplace=True)
     (downsample): Sequential(
       (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
       

In [32]:
inp = torch.ones(2,3,224,224)
for i, l in enumerate(layers):
    out = l(inp)
    print(f'layer_{i} shape: {out.shape}')
    inp = out

layer_0 shape: torch.Size([2, 64, 56, 56])
layer_1 shape: torch.Size([2, 256, 56, 56])
layer_2 shape: torch.Size([2, 512, 28, 28])
layer_3 shape: torch.Size([2, 1024, 14, 14])
layer_4 shape: torch.Size([2, 2048, 7, 7])


In [33]:
 def decoder_blk(self, inp_filt):
        conv_x_2x = nn.Sequential(
            conv_block(inp_filt*2, inp_filt),
            conv_block(inp_filt, inp_filt),
            nn.ConvTranspose2d(inp_filt, int(inp_filt/2), (2,2), 2)
            )
        return conv_x_2x

In [34]:
class Bro_RESUNET(nn.Module):
    def __init__(self, ifilt=3, nf=64):
        super().__init__()
        self.model = torch.hub.load("pytorch/vision", "resnet50", weights="ResNet50_Weights.IMAGENET1K_V1")
        d_f = [2048, 1024, 512, 256, 64]
        self.layer_0 = nn.Sequential(  
            self.model.conv1, #x/2
            self.model.bn1,
            self.model.relu,
            self.model.maxpool #x/4, 64
        )
        self.layer_1 = self.model.layer1 #x/4, 256
        self.layer_2 = self.model.layer2 #x/8, 512
        self.layer_3 = self.model.layer3 #x/16, 1024
        self.layer_4_bneck = self.model.layer4 #x/32, 2048

        self.bneck_x_2x = nn.ConvTranspose2d(d_f[0], d_f[1], (2,2), 2) #x/16, 2048, 1024

        self.dec_blk_layer_3 = self.decoder_blk(d_f[1]) #2048, 1024, 512
        self.dec_blk_layer_2 = self.decoder_blk(d_f[2]) #1024, 512, 256
        self.dec_blk_layer_1 = nn.Sequential(conv_block(d_f[3]*2, d_f[3]),
                                               conv_block(d_f[3], d_f[4]) 
                                              ) #512, 256, 64
      
        self.final_conv = nn.Sequential(
            nn.ConvTranspose2d(d_f[4]*2, d_f[4], (2,2), 2), #112
            nn.ConvTranspose2d(d_f[4], 1, (2,2), 2), #224 
            nn.Sigmoid()
            )

    def decoder_blk(self, inp_filt):
        conv_x_2x = nn.Sequential(
            conv_block(inp_filt*2, inp_filt),
            conv_block(inp_filt, inp_filt),
            nn.ConvTranspose2d(inp_filt, int(inp_filt/2), (2,2), 2)
            )
        return conv_x_2x

    def forward(self, x):
        s1 = self.layer_0(x)  #224, 56
        s2 = self.layer_1(s1) #56, 56
        s3 = self.layer_2(s2) #56, 28
        s4 = self.layer_3(s3) #28, 14

        out = self.layer_4_bneck(s4) #14, 7
        out = self.bneck_x_2x(out) #7, 14

        out = torch.cat((s4, out), 1) 
        out = self.dec_blk_layer_3(out) #14, 28
        out = torch.cat((s3, out), 1)
        out = self.dec_blk_layer_2(out) #28, 56
        out = torch.cat((s2, out), 1)
        out = self.dec_blk_layer_1(out) #56, 56
        out = torch.cat((s1, out), 1)
        out = self.final_conv(out) #56, 112, 224
        
        return out

In [35]:
model = Bro_RESUNET()

Using cache found in /home/ec2-user/.cache/torch/hub/pytorch_vision_main


In [36]:
inp = torch.ones(3, 3, 224, 224)
inp.shape

torch.Size([3, 3, 224, 224])

In [37]:
out = model(inp)

In [38]:
out.shape

torch.Size([3, 1, 224, 224])

In [39]:
model

Bro_RESUNET(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
        

In [40]:
summary(model, input_size=(1,3,224,224))

Layer (type:depth-idx)                   Output Shape              Param #
Bro_RESUNET                              [1, 1, 224, 224]          2,049,000
├─Sequential: 1-1                        [1, 64, 56, 56]           --
│    └─Conv2d: 2-1                       [1, 64, 112, 112]         9,408
│    └─BatchNorm2d: 2-2                  [1, 64, 112, 112]         128
│    └─ReLU: 2-3                         [1, 64, 112, 112]         --
│    └─MaxPool2d: 2-4                    [1, 64, 56, 56]           --
├─Sequential: 1-2                        [1, 256, 56, 56]          --
│    └─Bottleneck: 2-5                   [1, 256, 56, 56]          --
│    │    └─Conv2d: 3-1                  [1, 64, 56, 56]           4,096
│    │    └─BatchNorm2d: 3-2             [1, 64, 56, 56]           128
│    │    └─ReLU: 3-3                    [1, 64, 56, 56]           --
│    │    └─Conv2d: 3-4                  [1, 64, 56, 56]           36,864
│    │    └─BatchNorm2d: 3-5             [1, 64, 56, 56]          